# 01 — Compreensão do Problema e Análise Exploratória**Tech Challenge Fase 2 — Wine Quality Classification**Cobre as **Etapas 1 e 2** do enunciado:1. Compreensão do problema, definição do alvo e binarização.2. EDA: distribuições, correlações **justificadas**, outliers e balanceamento de classes.> Responsável: _(EDA & Qualidade de Dados)_

In [ ]:
import syssys.path.append("..")import matplotlib.pyplot as pltimport numpy as npimport pandas as pdimport seaborn as snsfrom src import data_loader as dlfrom src import preprocessing as pppd.set_option("display.max_columns", None)sns.set_theme(style="whitegrid")plt.rcParams["figure.figsize"] = (9, 5)

---## Etapa 1 — Compreensão do problema**Pergunta de negócio:** é possível prever a qualidade final de um vinho a partir dosindicadores físico-químicos medidos durante a produção?**Variável alvo:** `high_quality` — binária, derivada de `quality`:- `1` → nota ≥ 7 (alta qualidade)- `0` → nota < 7 (baixa/média qualidade)

In [ ]:
df_raw = dl.load_raw()   # winequality-red.csv por padrao; veja data/raw/README.mdprint("Formato:", df_raw.shape)df_raw.head()

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe().T

### Criação da variável alvo binária

In [ ]:
df = dl.add_binary_target(df_raw)df[["quality", "high_quality"]].head(10)

In [ ]:
# Como a nota original se distribui, e onde cai o corte em 7fig, ax = plt.subplots()sns.countplot(data=df, x="quality", hue="high_quality", palette="RdBu", ax=ax)ax.set_title("Distribuição das notas de qualidade e o corte em 7")ax.set_xlabel("Nota atribuída pelos especialistas")ax.set_ylabel("Quantidade de amostras")plt.savefig("../results/figures/distribuicao_notas.png", dpi=150, bbox_inches="tight")plt.show()

---## Etapa 2 — Análise Exploratória de Dados### 2.1 Balanceamento das classes> ⚠️ **Ponto crítico do case.** Se a classe "alta qualidade" for minoria, acurácia deixa de> ser métrica útil: um modelo que chuta sempre "baixa qualidade" acerta a maioria e não serve> para nada. Registre o percentual aqui e retome esse número na apresentação.

In [ ]:
balanco = dl.class_balance(df["high_quality"])display(balanco)fig, ax = plt.subplots(figsize=(5, 4))sns.barplot(x=balanco.index.map({0: "Baixa/Média", 1: "Alta"}),            y=balanco["quantidade"], palette=["#9CA3AF", "#7C3A4E"], ax=ax)for i, v in enumerate(balanco["quantidade"]):    ax.text(i, v, f'{v}\n({balanco["percentual"].iloc[i]}%)', ha="center", va="bottom")ax.set_title("Balanceamento das classes")ax.set_ylabel("Amostras")plt.savefig("../results/figures/balanceamento_classes.png", dpi=150, bbox_inches="tight")plt.show()

**Interpretação:** _escreva aqui em uma frase o grau de desbalanceamento e aconsequência metodológica (usar `class_weight='balanced'`, estratificar o split, decidir porF1/Recall/ROC-AUC em vez de acurácia)._

### 2.2 Qualidade dos dados: faltantes e duplicados

In [ ]:
display(pp.missing_report(df))print("Linhas duplicadas:", pp.duplicates_report(df))

### 2.3 Distribuição das variáveis

In [ ]:
num_cols = [c for c in df.columns if c not in ("quality", "high_quality")]df[num_cols].hist(bins=30, figsize=(15, 10), color="#7C3A4E", edgecolor="white")plt.suptitle("Distribuição das variáveis físico-químicas", y=1.01)plt.tight_layout()plt.savefig("../results/figures/distribuicoes.png", dpi=150, bbox_inches="tight")plt.show()

**Interpretação:** _quais variáveis são assimétricas? Alguma tem cauda longa à direita(ex.: açúcar residual, cloretos)? Isso justifica padronização na Etapa 3._

### 2.4 Como cada variável se comporta entre as duas classes

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(15, 14))for ax, col in zip(axes.ravel(), num_cols):    sns.boxplot(data=df, x="high_quality", y=col, palette=["#9CA3AF", "#7C3A4E"], ax=ax)    ax.set_xticklabels(["Baixa/Média", "Alta"])    ax.set_xlabel("")for ax in axes.ravel()[len(num_cols):]:    ax.axis("off")plt.suptitle("Variáveis físico-químicas por classe de qualidade", y=1.00)plt.tight_layout()plt.savefig("../results/figures/boxplots_por_classe.png", dpi=150, bbox_inches="tight")plt.show()

### 2.5 Correlações> 📌 O enunciado exige **justificar cada correlação relevante**. Não basta o heatmap:> cada par forte precisa de uma explicação enológica. Use a lista abaixo como ponto de partida> e confirme com os números do seu dataset.

In [ ]:
corr = df[num_cols + ["quality"]].corr()fig, ax = plt.subplots(figsize=(11, 9))mask = np.triu(np.ones_like(corr, dtype=bool))sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0,            square=True, linewidths=.5, ax=ax)ax.set_title("Matriz de correlação")plt.savefig("../results/figures/matriz_correlacao.png", dpi=150, bbox_inches="tight")plt.show()

In [ ]:
# Correlação de cada variável com a nota de qualidade, em ordemcorr_alvo = corr["quality"].drop("quality").sort_values(ascending=False)display(corr_alvo.to_frame("correlacao_com_quality").round(3))

#### Justificativas (preencher com os valores do seu dataset)| Par | Correlação | Justificativa enológica ||---|---|---|| `alcohol` × `quality` | _valor_ | Vinhos mais alcoólicos vêm de uvas mais maduras e concentradas; corpo e estrutura são premiados na análise sensorial. || `volatile acidity` × `quality` | _valor_ | Acidez volátil alta é **defeito**: é o ácido acético, o "cheiro de vinagre". Quanto maior, pior a nota. || `density` × `alcohol` | _valor_ | O álcool é menos denso que a água; mais álcool ⇒ menor densidade. Relação físico-química direta. || `free SO2` × `total SO2` | _valor_ | O livre é subconjunto do total — correlação estrutural, atenção à multicolinearidade. || `fixed acidity` × `pH` | _valor_ | Mais ácido ⇒ pH menor. Relação inversa por definição. || `citric acid` × `fixed acidity` | _valor_ | O ácido cítrico é um dos componentes da acidez fixa. || _outro_ | | |**Multicolinearidade detectada:** _listar os pares acima de |0.7| e decidir se alguma variávelserá removida na Etapa 3._

### 2.6 Outliers

In [ ]:
outliers = pp.outlier_report(df[num_cols])display(outliers)

**Decisão sobre outliers:** _manter, remover ou winsorizar? Justifique. Em dadosfísico-químicos, valores extremos costumam ser reais (vinhos atípicos) e não erros de digitação —removê-los sem critério joga fora informação legítima._

---## Síntese da EDA (insumo direto da apresentação executiva)1. **Balanceamento:** _..._2. **Variáveis mais associadas à qualidade:** _..._3. **Variáveis que indicam defeito:** _..._4. **Problemas de qualidade de dados:** _..._5. **Decisões que serão levadas para o pré-processamento:** _..._

In [ ]:
# Base com o alvo criado, para o notebook 02df.to_csv("../data/processed/wine_com_alvo.csv", index=False)print("Salvo em data/processed/wine_com_alvo.csv")